> **Public-release version:** Private animal-level inputs and source-derived observations have been removed from the notebook. The original modelling and plotting cells are retained. See the repository README for data requirements.

# NB17 v1.1 — PE and GI population composition with uncertainty table

**Purpose:** Prepare one black-and-white two-panel composition figure and an accompanying table of exact posterior median fractions and 95% credible intervals. No Bayesian fitting is performed.

**Sources:** Published aggregate summaries in `figures/`; no animal-level data are embedded.

**Caveat:** Marginal posterior component medians need not sum to exactly 100%; the plot normalizes them for illustration only. The table preserves the unmodified medians and credible intervals. Estimates are exploratory, with later PE sampling not time-matched and repeated GI animal observations. This notebook does not alter the manuscript.

In [ ]:
#@title NB17.1 — Imports and output folder
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display
import base64, zlib, hashlib, io
OUTPUT=Path.cwd()/'NB17_PE_GI_Figure_and_Table'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    home=Path('/content/drive/MyDrive')
    target=home/'Transmid_Animal_Bayesian_Project'
    matches=[target] if target.is_dir() else [x for x in home.rglob('Transmid_Animal_Bayesian_Project') if x.is_dir()]
    if len(matches)==1:
        OUTPUT=matches[0]/'06_Results'/'NB17_PE_GI_Figure_and_Table'
except ImportError:
    pass
OUTPUT.mkdir(parents=True,exist_ok=True)
print('Output:',OUTPUT)

In [ ]:
#@title NB17.2 — Load published aggregate PE/GI Bayesian summaries
from pathlib import Path
import pandas as pd
GROUP_SUMMARY=Path('figures')/'NB17_Unmodified_Posterior_Summaries.csv'
if not GROUP_SUMMARY.is_file():
    GROUP_SUMMARY=Path('..')/'figures'/'NB17_Unmodified_Posterior_Summaries.csv'
if not GROUP_SUMMARY.is_file():
    raise FileNotFoundError('Group-level NB17 summary CSV is required. Run from repository root or notebooks/.')
summary=pd.read_csv(GROUP_SUMMARY)
assert len(summary)==28 and (summary.q025<=summary['median']).all() and (summary['median']<=summary.q975).all()
print('Loaded 28 published aggregate PE/GI composition rows (no individual observations).')


In [ ]:
#@title NB17.3 — Exact median and 95% credible interval table
names={'R':'R','D':'D','T1':'T1','Ts':'T2−T1'}
def pct(x):
    v=100*float(x)
    if v<0.01:return '<0.01'
    if v<0.1:return f'{v:.2f}'
    return f'{v:.1f}'
rows=[]
for study in ['PE','GI']:
    for idx in range(4):
        sub=summary[(summary.study==study)&(summary.group_index==idx)].set_index('component')
        row={'Study':study,'Group':sub.group_name.iloc[0]}
        for comp in ['R','D','T1','Ts']:
            if comp not in sub.index: row[names[comp]]='—'
            else:
                val=sub.loc[comp]
                row[names[comp]]=f"{pct(val['median'])} ({pct(val.q025)}–{pct(val.q975)})"
        rows.append(row)
summary_table=pd.DataFrame(rows)
display(summary_table)
summary_table.to_csv(OUTPUT/'Table_PE_GI_Population_Medians_95CrI.csv',index=False)
summary.to_csv(OUTPUT/'NB17_Unmodified_Posterior_Summaries.csv',index=False)

In [ ]:
#@title NB17.4 — Black-and-white two-panel composition figure
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'savefig.facecolor':'white'})
style={'R':dict(color='white',edgecolor='black',hatch=''),
       'D':dict(color='#e3e3e3',edgecolor='black',hatch='///'),
       'T1':dict(color='#999999',edgecolor='black',hatch='xx'),
       'Ts':dict(color='#262626',edgecolor='black',hatch='')}
comps={'PE':['R','T1','Ts'],'GI':['R','D','T1','Ts']}
short={'R':'R','D':'D','T1':'T1','Ts':'T2−T1'}
fig,axes=plt.subplots(1,2,figsize=(13.6,5.9),layout='constrained')
for ax,study in zip(axes,['PE','GI']):
    med=[]
    for i in range(4):
        sub=summary[(summary.study==study)&(summary.group_index==i)].set_index('component')
        vals=np.array([sub.loc[c,'median'] for c in comps[study]])
        med.append(vals/vals.sum())
    med=np.array(med)
    x=np.arange(4); bottom=np.zeros(4)
    for j,c in enumerate(comps[study]):
        height=100*med[:,j]
        ax.bar(x,height,bottom=bottom,width=.73,linewidth=.8,zorder=3,**style[c])
        for xx,yy,bb in zip(x,height,bottom):
            if yy>=10:
                ax.text(xx,bb+yy/2,f'{yy:.0f}%',ha='center',va='center',fontsize=9,
                        color='white' if c=='Ts' else 'black',weight='bold')
        bottom+=height
    assert np.allclose(bottom,100)
    labels=['6 h\nTarget','6 h\nNon-target','16–20 h\nTarget','24 h\nNon-target'] if study=='PE' else ['Day 10\nTarget','Day 10\nNon-target','Day 14\nTarget','Day 14\nNon-target']
    ax.set_xticks(x,labels);ax.set_xlim(-.55,3.55);ax.set_ylim(0,100)
    ax.set_yticks([0,25,50,75,100]);ax.grid(axis='y',alpha=.3,zorder=0,color='black',lw=.4)
    ax.spines[['top','right']].set_visible(False)
    ax.set_title('A  Peritoneal (PE)' if study=='PE' else 'B  Gastrointestinal (GI)',loc='left',fontsize=12,weight='bold')
    if study=='PE': ax.set_ylabel('Estimated population composition (%)')
handles=[Patch(label=short[c],linewidth=.8,**style[c]) for c in ['R','D','T1','Ts']]
fig.legend(handles=handles,ncol=4,frameon=False,loc='upper center',bbox_to_anchor=(.5,1.055))
fig.suptitle('Population composition after Transmid delivery',y=1.12,fontsize=14,weight='bold')
fig.text(.5,-.045,'Marginal medians rescaled to sum to 100% for display only; unmodified medians and 95% credible intervals are tabulated.',ha='center',fontsize=8.7)
fig.text(.5,-.085,'PE later groups are not time-matched; GI animals were measured repeatedly. Exploratory Bayesian estimates.',ha='center',fontsize=8.3)
fig.savefig(OUTPUT/'Figure_5_PE_GI_Composition_2Panel_BW.png',dpi=350,bbox_inches='tight')
fig.savefig(OUTPUT/'Figure_5_PE_GI_Composition_2Panel_BW.svg',bbox_inches='tight')
plt.show()